# 📊 Trading Forecasting - Exploração de Dados

Este notebook realiza a exploração inicial dos dados de propostas de fornecedores para análise de descontos aplicáveis.

## Objetivos:
- Carregar e examinar os dados
- Análise exploratória dos dados (EDA)
- Identificar padrões e insights
- Preparar dados para modelagem

In [ ]:
# Importações necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configurações de visualização
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Carregamento dos Dados

In [ ]:
# Carregar dados (ajustar caminho conforme necessário)
# Para uso local
data_path = "../data/historico_propostas.xlsx"

try:
    df = pd.read_excel(data_path)
    print(f"✅ Dados carregados com sucesso! Shape: {df.shape}")
except FileNotFoundError:
    print("❌ Arquivo não encontrado. Verifique o caminho dos dados.")
    # Criar dados de exemplo para demonstração
    np.random.seed(42)
    
    df = pd.DataFrame({
        'CodigoMaterial': np.random.choice(['10000', '10001', '10002', '10003'], 1000),
        'Fornecedor': np.random.choice(['Alpha Ltda', 'Beta Corp', 'Gamma SA', 'Delta Inc'], 1000),
        'Quantidade': np.random.randint(50, 500, 1000),
        'PrazoEntrega(dias)': np.random.randint(5, 30, 1000),
        'CondicoesPagamento': np.random.choice(['À vista', '30 dias', '60 dias'], 1000),
        'FreteIncluso': np.random.choice(['Sim', 'Não'], 1000),
        'ValidadeProposta(dias)': np.random.randint(15, 60, 1000),
        'PrecoUnitario(R$)': np.random.uniform(100, 500, 1000),
        'DescontoAplicavel(%)': np.random.uniform(0, 25, 1000)
    })
    print(f"📋 Usando dados de exemplo. Shape: {df.shape}")

## 2. Visão Geral dos Dados

In [ ]:
# Informações básicas do dataset
print("=== INFORMAÇÕES BÁSICAS ===\n")
print(f"Dimensões: {df.shape}")
print(f"Colunas: {list(df.columns)}")
print("\n=== TIPOS DE DADOS ===\n")
print(df.dtypes)
print("\n=== VALORES AUSENTES ===\n")
print(df.isnull().sum())

In [ ]:
# Primeiras linhas dos dados
print("=== PRIMEIRAS 5 LINHAS ===\n")
df.head()

In [ ]:
# Estatísticas descritivas
print("=== ESTATÍSTICAS DESCRITIVAS ===\n")
df.describe()

## 3. Análise por Material

In [ ]:
# Distribuição de propostas por material
material_counts = df['CodigoMaterial'].value_counts()
print("=== DISTRIBUIÇÃO POR MATERIAL ===\n")
print(material_counts)

# Visualização
plt.figure(figsize=(10, 6))
material_counts.plot(kind='bar')
plt.title('Número de Propostas por Código de Material')
plt.xlabel('Código do Material')
plt.ylabel('Número de Propostas')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Análise de Fornecedores

In [ ]:
# Análise de fornecedores
print("=== ANÁLISE DE FORNECEDORES ===\n")
fornecedor_stats = df.groupby('Fornecedor').agg({
    'DescontoAplicavel(%)': ['count', 'mean', 'std'],
    'PrecoUnitario(R$)': ['mean', 'std'],
    'PrazoEntrega(dias)': 'mean'
}).round(2)

fornecedor_stats.columns = ['Num_Propostas', 'Desconto_Medio', 'Desconto_Std', 
                           'Preco_Medio', 'Preco_Std', 'Prazo_Medio']
print(fornecedor_stats)

In [ ]:
# Visualização de descontos por fornecedor
plt.figure(figsize=(12, 8))
sns.boxplot(data=df, x='Fornecedor', y='DescontoAplicavel(%)')
plt.title('Distribuição de Descontos por Fornecedor')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Análise de Correlações

In [ ]:
# Selecionar apenas colunas numéricas para correlação
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlation_matrix = df[numeric_cols].corr()

# Heatmap de correlações
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Matriz de Correlação - Variáveis Numéricas')
plt.tight_layout()
plt.show()

print("\n=== CORRELAÇÕES COM DESCONTO APLICÁVEL ===\n")
desconto_corr = correlation_matrix['DescontoAplicavel(%)'].sort_values(ascending=False)
print(desconto_corr)

## 6. Análise de Variáveis Categóricas

In [ ]:
# Análise por condições de pagamento
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Condições de pagamento
sns.boxplot(data=df, x='CondicoesPagamento', y='DescontoAplicavel(%)', ax=axes[0,0])
axes[0,0].set_title('Desconto por Condição de Pagamento')
axes[0,0].tick_params(axis='x', rotation=45)

# Frete incluso
sns.boxplot(data=df, x='FreteIncluso', y='DescontoAplicavel(%)', ax=axes[0,1])
axes[0,1].set_title('Desconto por Frete Incluso')

# Distribuição de quantidade
df['Quantidade'].hist(bins=30, ax=axes[1,0])
axes[1,0].set_title('Distribuição de Quantidades')
axes[1,0].set_xlabel('Quantidade')

# Relação quantidade vs desconto
axes[1,1].scatter(df['Quantidade'], df['DescontoAplicavel(%)'], alpha=0.6)
axes[1,1].set_title('Quantidade vs Desconto')
axes[1,1].set_xlabel('Quantidade')
axes[1,1].set_ylabel('Desconto Aplicável (%)')

plt.tight_layout()
plt.show()

## 7. Insights e Conclusões da EDA

In [ ]:
# Resumo de insights
print("=== INSIGHTS DA ANÁLISE EXPLORATÓRIA ===\n")

print("1. DISTRIBUIÇÃO DOS DADOS:")
print(f"   - Total de registros: {len(df):,}")
print(f"   - Número de materiais únicos: {df['CodigoMaterial'].nunique()}")
print(f"   - Número de fornecedores únicos: {df['Fornecedor'].nunique()}")
print(f"   - Desconto médio geral: {df['DescontoAplicavel(%)'].mean():.2f}%")

print("\n2. VARIÁVEIS MAIS CORRELACIONADAS COM DESCONTO:")
for var, corr in desconto_corr.items():
    if var != 'DescontoAplicavel(%)' and abs(corr) > 0.1:
        print(f"   - {var}: {corr:.3f}")

print("\n3. RECOMENDAÇÕES PARA MODELAGEM:")
print("   ✅ Usar Random Forest para lidar com variáveis categóricas")
print("   ✅ Criar modelos específicos por material (se houver dados suficientes)")
print("   ✅ Considerar interações entre fornecedor e condições de pagamento")
print("   ✅ Avaliar outliers em preços e descontos")

## 8. Preparação para Próxima Etapa

In [ ]:
# Salvar dados preparados para treinamento
output_path = "../data/dados_preparados.csv"
df.to_csv(output_path, index=False)
print(f"✅ Dados salvos em: {output_path}")

# Salvar resumo da EDA
eda_summary = {
    'total_records': len(df),
    'unique_materials': df['CodigoMaterial'].nunique(),
    'unique_suppliers': df['Fornecedor'].nunique(),
    'avg_discount': df['DescontoAplicavel(%)'].mean(),
    'correlations': desconto_corr.to_dict()
}

import json
with open('../data/eda_summary.json', 'w') as f:
    json.dump(eda_summary, f, indent=2)

print("✅ Resumo da EDA salvo em: ../data/eda_summary.json")
print("\n🎯 Próximo passo: Execute o notebook 02_model_training.ipynb")